在这个案例中，我们将基于Python与面向对象编程的思想来复现一个历史上著名的博弈论实验，即“阿克塞尔罗德竞赛（Axelrod Touraments）”。

请先观看背景介绍视频：[为什么一国涨了关税，另一国必须对等报复？【差评君】](https://www.bilibili.com/video/BV1rR57zzEoC)

In [1]:
import pandas as pd
import random
from typing import Any, List, Tuple, Dict
from dataclasses import dataclass

在阿克塞尔罗德竞赛（Axelrod Touraments）中，来自各学科的专家提交的策略程序在一个循环锦标赛中两两对战，通过多轮重复博弈，揭示了合作、报复、信任等行为在长期互动中的演化机制。

在本项目中，我们将构建一个简化版本的 Axelrod Touraments 系统，实现多种策略之间的模拟对战，并进行得分的统计和排名。

在该博弈论实验中，涉及到玩家、策略、收益矩阵等几个重要元素，可以抽象成不同的类。

# 收益矩阵

视频中的收益矩阵如下方所示
```
Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (3, 3)  (0, 5)
          D     (5, 0)  (1, 1)
```
其中C代表合作（cooperation），D代表背叛（defect）。矩阵的每个格子对应一种可能的博弈结果，格子里以二元组(num1,num2)的形式，记录了那种博弈结果中两个玩家各自的收益。其中，约定num1是player1（行玩家）的收益，num2是player2（列玩家）的收益。
这种记法可以被总结为“左行右列”。这种记法下，分析每个玩家的最优策略的思路可以被总结为“行玩家按行比收益，列玩家按列比收益”。

注意到囚徒困境的收益矩阵可以被写为如下更一般的形式
```
payoff_matrix = {
    ('C', 'C'): (R, R),
    ('C', 'D'): (S, T),
    ('D', 'C'): (T, S),
    ('D', 'D'): (P, P),
    }
```
其中，
+ R为 player 1 在payoff_matrix中('C', 'C')格子的得分，该字母是Reward（奖励）的简称，表示双方都选择合作时的收益；
+ P为 player 1 在payoff_matrix中('D', 'D')格子的得分，该字母是Punishment（惩罚）的简称，表示双方都选择背叛时的收益；
+ S为 player 1 在payoff_matrix中('C', 'D')格子的得分，该字母是Sucker's payoff（傻瓜的收益）的简称，表示一方选择合作而另一方选择背叛时，合作方的收益；
+ T为 player 1 在payoff_matrix中('D', 'C')格子的得分，是Temptation的简称，该字母是Temptation（诱惑）的简称，表示一方选择背叛而另一方选择合作时，背叛方的收益。

囚徒困境的核心设定可以一般性地描述为 **T > R > P > S**。上述记号方式的来源可参考[1]和[2]。

[1] https://en.wikipedia.org/wiki/Prisoner's_dilemma

[2] Press, W. H., & Dyson, F. J. (2012). Iterated Prisoner’s Dilemma contains strategies that dominate any evolutionary opponent. Proceedings of the National Academy of Sciences, 109(26), 10409–10413. https://doi.org/10.1073/pnas.1206569109

基于上述背景，可以定义一个名为 Payoff 的类，用来存储双方采取不同行动时的收益。

In [2]:
# 麻烦的写法
class Payoff:
    
    def __init__(self, R: int=3, P: int=1, S: int=0, T: int=5) -> None:
        assert T > R > P > S
        self.R = R
        self.P = P
        self.S = S
        self.T = T
        self.matrix = {
            ('C', 'C'): (self.R, self.R),
            ('C', 'D'): (self.S, self.T),
            ('D', 'C'): (self.T, self.S),
            ('D', 'D'): (self.P, self.P),
        }
payoff = Payoff()
payoff.R

3

In [3]:
# 简洁的写作，使用dataclass
@dataclass
class Payoff:
    
    R: int = 3
    P: int = 1
    S: int = 0
    T: int = 5
    
payoff = Payoff()
payoff.R

3

In [4]:
# 使用dataclass，进一步完善功能
@dataclass
class Payoff:
    
    verbose: bool = False
    R: int = 3
    P: int = 1
    S: int = 0
    T: int = 5
    
    def __post_init__(self):
        assert self.T > self.R > self.P > self.S #检查 T > R > P > S 是否成立
        self.matrix = self.derive_payoff_matrix(self.R, self.P, self.S, self.T)
    
    def __repr__(self):      
        matrix_str = f"Payoff Matrix:\n" \
                     f"                   player 2 \n" \
                     f"                   C       D\n" \
                     f"player 1  C     {self.matrix[('C', 'C')]}  {self.matrix[('C', 'D')]}\n" \
                     f"          D     {self.matrix[('D', 'C')]}  {self.matrix[('D', 'D')]}"
        return matrix_str
    
    def __setattr__(self, name: str, value: Any):
        """通过属性赋值拦截来实现两个功能
        1. 属性验证：当被更新的属性是R、P、T、S中的一个时，验证更新是否合理，
           即更新后是否仍满足条件 T > R > P > S，如发现不满足，则自动撤销更新恢复旧的取值
        2. 联动属性的自动更新：一但R、P、T、S中的任意一个被成功更新后，重新计算属性matrix的取值
        """
        if self.verbose: print(f"executing `self.{name}={value}`")
        if (name in ['R', 'P', 'S', 'T']) and (name in vars(self)):
            old_value = getattr(self, name)
            if self.verbose: print(f"overwriting attribute value from {name}={old_value} to {name}={value}")
            try:
                super().__setattr__(name, value)
                # 验证更新是否合理
                assert self.T > self.R > self.P > self.S
                # 强制触发matrix的取值更新
                if self.verbose: print(f"updating `self.matrix`")
                self.matrix = self.derive_payoff_matrix(self.R, self.P, self.S, self.T)
            except AssertionError:
                if self.verbose: print(f"The update failed because setting {name}={value} violate the constraint T>R>P>S({self.T}>{self.R}>{self.P}>{self.S}). No change occurred!")
                super().__setattr__(name, old_value)
        else:
            super().__setattr__(name, value)
            
    @staticmethod
    def derive_payoff_matrix(R, P, S, T):
        return {
            ('C', 'C'): (R, R),
            ('C', 'D'): (S, T),
            ('D', 'C'): (T, S),
            ('D', 'D'): (P, P),
        } 

In [5]:
payoff = Payoff(R=3, P=1, S=0, T=5)
payoff

Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (3, 3)  (0, 5)
          D     (5, 0)  (1, 1)

In [6]:
payoff = Payoff(R=3, P=1, S=0, T=5, verbose=True)
print(payoff)
payoff.R = 0 # invalid update, see what happens?
print(payoff) # should stay the same
payoff.R = 4 # valid update
print(payoff) # should be different

executing `self.R=3`
executing `self.P=1`
executing `self.S=0`
executing `self.T=5`
executing `self.matrix={('C', 'C'): (3, 3), ('C', 'D'): (0, 5), ('D', 'C'): (5, 0), ('D', 'D'): (1, 1)}`
Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (3, 3)  (0, 5)
          D     (5, 0)  (1, 1)
executing `self.R=0`
overwriting attribute value from R=3 to R=0
The update failed because setting R=0 violate the constraint T>R>P>S(5>0>1>0). No change occurred!
Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (3, 3)  (0, 5)
          D     (5, 0)  (1, 1)
executing `self.R=4`
overwriting attribute value from R=3 to R=4
updating `self.matrix`
executing `self.matrix={('C', 'C'): (4, 4), ('C', 'D'): (0, 5), ('D', 'C'): (5, 0), ('D', 'D'): (1, 1)}`
Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (4, 4)  (0, 5)
          D     (5, 0)  (1, 1)


# 玩家基类

在该实验中，玩家可以采取不同的策略，但具有相同的基本属性，可以定义一个玩家基类，来刻画玩家的基本行为，记录玩家信息及其行动历史。

对于每个玩家，应当记录的基本信息包括：

- 玩家的名称（name）
- 玩家的行动历史（history）
- 玩家的合作次数（cooperations）
- 玩家的背叛次数（defections）

对于每个玩家，可以实现的方法包括：
1. move：根据对手的上一步行动，返回自己当前回合的行动（'C' 或 'D'），由于不同玩家的策略不同，可以在子类中重写，否则抛出 NotImplementedError
2. record_moves：记录玩家一次行动，更新 history、cooperations、defections
3. reset：重置玩家的行动历史，清空 history、cooperations、defections
4. `__repr__`: 返回玩家对象的字符串表示。

In [7]:
class Player:
    
    def __init__(self, name: str):
        self.name: str = name
        self.history: List[str] = []
        self.cooperations: int = 0
        self.defections: int = 0

    def move(self, opponent) -> str:
        """返回 'C'（合作）或 'D'（背叛）"""
        raise NotImplementedError("子类必须实现 move 方法")

    def record_moves(self, action: str) -> None:
        self.history.append(action)
        if action == 'C':
            self.cooperations += 1
        elif action == 'D':
            self.defections += 1
        else:
            raise ValueError(f"Invalid action: {action}")

    def reset(self) -> None:
        self.history.clear()
        self.cooperations = 0
        self.defections = 0

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}('{self.name}')"


In [8]:
Player_a=Player("a")
print(Player_a)

Player('a')


## 不同策略的玩家

采取不同策略的玩家，在move方法中，会根据当前的状态，选择不同的行动。可以通过重写move方法来实现。

### 总是合作和总是背叛
最简单的策略就是，不管对手如何选择，始终选择合作或背叛。

In [9]:
class AlwaysCooperatePlayer(Player):
    def move(self, opponent: Player) -> str:
        return 'C'


class AlwaysDefectPlayer(Player):
    def move(self, opponent: Player) -> str:
        return 'D'

### 随机策略

还可以不管对手如何选择，都随机选择合作或背叛。

In [10]:
class RandomPlayer(Player):
    def move(self, opponent: Player) -> str:
        return random.choice(['C', 'D'])

### 唐宁策略
唐宁策略（Downing Strategy）是一种基于贝叶斯估计 + 最优响应的囚徒困境博弈策略，由 凯文·唐宁（Kevin Downing）提出。

核心思路：

记“C_s”代表事件“上一轮自己选择合作”，“D_s”代表事件“上一轮自己选择背叛”，

“C_o”代表事件“这一轮对手选择合作”，“D_o”代表事件“这一轮对手选择背叛”。

另记：
+ alpha = P(C_o | C_s) 表示在自己上一轮选择合作的情况下，对手这一轮选择合作的概率，
+ beta =  P(C_o | D_s) 表示在自己上一轮选择背叛的情况下，对手这一轮选择合作的概率。

则有
+ 1-alpha = P(D_o | C_s) 表示在自己上一轮选择合作的情况下，对手这一轮选择背叛的概率，
+ 1-beta = P(D_o | D_s)  表示在自己上一轮选择背叛的情况下，对手这一轮选择背叛的概率。
          
根据收益矩阵中：R表示双方都选择合作时的收益；P表示双方都选择背叛时的收益；
S表示一方选择合作而另一方选择背叛时，合作方的收益；T表示一方选择背叛而另一方选择合作时，背叛方的收益。
则不难有如下推论：
+ 长期来看，如果自己一直选择合作，平均每轮的期望收益为 E_C = alpha * R + (1 - alpha) * S
+ 长期来看，如果自己一直选择背叛，平均每轮的期望收益为 E_D = beta  * T + (1 - beta) *  P
        
在第N轮，玩家根据历史观测到的信息，得到alpha和beta的最新估计值，并执行如下策略：
+ 若 E_C > E_D，则选择合作；
+ 若 E_C < E_D，则选择背叛；
+ 若 E_C = E_D，则选择跟上一轮自身策略相反的策略。
        
注意到，根据囚徒困境的核心设定 T > R > P > S，
  如果在第1轮定义 alpha = beta = 0.5，则有 E_C = (R+S)/2 < (T+P)/2 = E_D，因此第1轮一定选择背叛。
          
进一步，如果在第1轮对方选择合作，则约定将beta更新为 beta = 1，而alpha保持估计值0.5不变。
  此时有 E_C = (R+S)/2 < T = E_D，因此第2轮一定还选择背叛。
反之，则更新beta的估计值为 beta = 0，此时有 E_C = (R+S)/2 且 E_D = P，需要进一步判断。


不同于前面的简单策略，在唐宁策略中，下一步行动，需要通过条件概率估计对手的合作倾向（alpha，beta），再基于收益矩阵计算长期期望得分，做出理性决策。

因此需要添加两个属性，来记录对手对于己方行动的回应，并在比赛过程中（move方法）进行更新，在比赛结束后进行重置（重载reset方法）。


In [11]:
class DowningPlayer(Player):
    
    def __init__(self, name: str, payoff: Payoff):
        super().__init__(name)
        self.payoff = payoff
        self.number_opponent_cooperations_in_response_to_C = 0
        self.number_opponent_cooperations_in_response_to_D = 0

    def move(self, opponent: Player) -> str:

        round_number = len(self.history) + 1
        
        if round_number == 1:
            return 'D'
        
        if round_number == 2:
            if opponent.history[-1] == 'C':
                self.number_opponent_cooperations_in_response_to_C += 1
            return 'D'
        
        if self.history[-2] == 'C' and opponent.history[-1] == 'C':
            self.number_opponent_cooperations_in_response_to_C += 1
        if self.history[-2] == 'D' and opponent.history[-1] == 'C':
            self.number_opponent_cooperations_in_response_to_D += 1

        # Adding 1 to cooperations for assumption that first opponent move
        # being a response to a cooperation.
        alpha = (self.number_opponent_cooperations_in_response_to_C /
                 (self.cooperations + 1))
        # Adding 2 to defections on the assumption that the first two
        # moves are defections, which may not be true in a noisy match
        beta = (self.number_opponent_cooperations_in_response_to_D /
                 max(self.defections, 2))
        
        E_C = alpha * self.payoff.R + (1 - alpha) * self.payoff.S
        E_D = beta *  self.payoff.T + (1 - beta) * self.payoff.P
        
        if E_C > E_D:
            return 'C'
        elif E_C < E_D:
            return 'D'
        else:
            return 'D' if self.history[-1] == 'C' else 'C'

    def reset(self) -> None:
        super().reset()
        self.number_opponent_cooperations_in_response_to_C = 0
        self.number_opponent_cooperations_in_response_to_D = 0


# Tournament锦标赛

模拟多个玩家（策略）**两两**对战若干轮，每轮都是某个player1对战某个player2的形式。我们需要为每个玩家记录其行动与得分，最终根据每位玩家的总分进行排名分析。

为了实现这些功能，我们设计一个类 Tournament 来：

- 管理玩家对战
- 控制比赛轮次
- 记录每场比赛的数据
- 输出分析结果和对战历史

该类的属性包括：

| 属性名             | 类型                                                   | 含义与功能说明                                     |
|--------------------|--------------------------------------------------------|----------------------------------------------------|
| `players`          | `List[Player]`                                         | 所有参赛选手（策略）列表                          |
| `payoff`           | `Payoff`                                               | 收益矩阵，决定行动得分规则                        |
| `rounds`           | `int`                                                  | 每场比赛的轮数                                     |
| `_results`         | `Dict[str, int]`                                       | 存储每个玩家的累计得分（按名称索引）              |
| `_match_results`   | `Dict[Tuple[str, str], int]`                           | 存储每组玩家的对战结果，即记录ID为`(player1, player2)`的比赛条目中，player1得到的分数 <br> （player2的分数记录在ID为`(player2, player1)`的比赛条目中）   |
| `_round_history`   | `Dict[Tuple[str, str], List[Tuple[str, str]]]`        | 存储每一组玩家在所有回合中的行为（如：`[('C','D'),('D','D'),...]`）     |


该类的方法包括：
- `_match(p1, p2)`：模拟两个玩家之间的多轮对战；每轮获取双方动作，更新得分；记录每一轮行为
- `run()`：循环锦标赛的主控制流程；每两位玩家进行一次 _match()；累加得分并记录比赛结果到 _results 和 _match_results
- `print_rankings()`：打印总排行榜（按平均得分降序）
- ` __call__()`：让 Tournament 实例可以像函数一样调用；内部自动调用 run() 和 print_rankings()
- `get_round_history(p1_name, p2_name)`：返回指定两位玩家对战的每一回合行为,用于复盘分析或可视化展示
- `get_match_results()`：返回一个对战得分矩阵，行列为玩家名，元素为 player1 的得分，每行计算“平均得分”列供分析排名


这些功能已在game.py中实现，这里直接调用展示结果，在作业中需要加以补充

In [12]:
from game import Tournament

# 主函数

现在，可以使用以上实现的类来模拟阿克塞尔罗德竞赛（Axelrod Touraments）

策略矩阵为R=3, P=1, S=0, T=10，参与者两两对决，进行200轮，并输出比赛结果。

In [13]:
#payoff = Payoff()
payoff = Payoff(R=3, P=1, S=0, T=5)
payoff

Payoff Matrix:
                   player 2 
                   C       D
player 1  C     (3, 3)  (0, 5)
          D     (5, 0)  (1, 1)

In [14]:
players = [
    AlwaysCooperatePlayer("AlwaysCooperate"),
    AlwaysDefectPlayer("AlwaysDefect"),
    DowningPlayer("Downing", payoff),
    RandomPlayer("Random"),
]

In [15]:
# random.seed(2025)
tournament = Tournament(players, payoff, rounds=200)
tournament()  # 直接调用 __call__ 方法

=== Axelrod Tournament Results ===
1. Downing                 : 507 points (on average)
2. AlwaysDefect            : 493 points (on average)
3. Random                  : 363 points (on average)
4. AlwaysCooperate         : 220 points (on average)


In [16]:
tmp_df = tournament.get_round_history('AlwaysDefect', 'Downing')
display(tmp_df)

,P1: AlwaysDefect,P2: Downing
1,D,D
2,D,D
3,D,D
4,D,D
5,D,D
...,...,...
196,D,D
197,D,D
198,D,D
199,D,D


In [17]:
# tournament.get_round_history('Downing', 'AlwaysDefect')

In [18]:
tournament.get_match_results()

,Downing,AlwaysDefect,Random,AlwaysCooperate,Average Score
Downing,200,200,630,998,507
AlwaysDefect,200,200,572,1000,493
Random,106,102,451,794,363
AlwaysCooperate,3,0,279,600,220


除以上实现的四种策略外，还有很多其他策略，已在strategy.py中实现，直接引入。

In [19]:
from strategy import *

In [20]:
players = [
    # AlwaysCooperatePlayer("AlwaysCooperate"),
    # AlwaysDefectPlayer("AlwaysDefect"),
    TitForTatPlayer("TitForTat"),
    TitForTwoTatsPlayer("TitForTwoTats"),
    DavisPlayer("Davis"),
    DowningPlayer("Downing", payoff),
    FeldPlayer("Feld"),
    GrudgerPlayer("Grudger"),
    RandomPlayer("Random"),
    ShubikPlayer("Shubik"),
]

In [21]:
# random.seed(2025)
tournament = Tournament(players, payoff, rounds=200)
tournament()  # 直接调用 __call__ 方法

=== Axelrod Tournament Results ===
1. Shubik                  : 552 points (on average)
2. TitForTat               : 538 points (on average)
3. TitForTwoTats           : 506 points (on average)
4. Davis                   : 506 points (on average)
5. Grudger                 : 505 points (on average)
6. Downing                 : 402 points (on average)
7. Feld                    : 376 points (on average)
8. Random                  : 305 points (on average)


In [22]:
# tmp_df = tournament.get_round_history('Feld', 'Shubik')
# display(tmp_df)

In [23]:
tournament.get_match_results()

,Shubik,TitForTat,TitForTwoTats,Davis,Grudger,Downing,Feld,Random,Average Score
Shubik,600,600,600,600,600,597,273,547,552
TitForTat,600,600,600,600,600,597,276,433,538
TitForTwoTats,600,600,600,600,600,286,376,387,506
Davis,600,600,600,600,600,208,237,599,506
Grudger,600,600,600,600,600,211,261,571,505
Downing,597,597,626,238,201,200,429,326,402
Feld,321,237,554,233,227,679,227,528,376
Random,225,452,632,144,108,101,366,414,305
